In [1]:
from langchain.tools import tool
from langchain_ollama import ChatOllama
import base64

C:\Users\Stefano\anaconda3\envs\GenAi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

In [5]:
load_dotenv(dotenv_path="setup.env",override=True)  

OLLAMA_SERVER = os.getenv("OLLAMA_SERVER",default="localhost:11434")
PROMPT_CREATOR_MODEL = os.getenv("PROMPT_CREATOR_MODEL",default="None")
SKETCH_DESCRIPTOR_MODEL = os.getenv("SKETCH_DESCRIPTOR_MODEL",default="None")
print(f"Ollama Server: {OLLAMA_SERVER}\nPrompt Creator Model: {PROMPT_CREATOR_MODEL}\nSketch Descriptor Model: {SKETCH_DESCRIPTOR_MODEL}")


Ollama Server: http://stefano-ideapad-pro-5:11434
Prompt Creator Model: llama3.2:3b
Sketch Descriptor Model: qwen3-vl:4b


In [6]:
sketch_descriptor_llm= ChatOllama(model=SKETCH_DESCRIPTOR_MODEL,base_url=OLLAMA_SERVER)

In [7]:
sketch_path = "sketch.jpg"

In [8]:

def encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [20]:

@tool("get_sketch_desc_prompt")
def get_sketch_desc_prompt(inputs: dict) -> str:
    """
    Get a description for the sketch
    
    Expected input:
    {
        "image_path": "...",
       
    }
    """
    
    prompt = """
Semantic description of the scene, in a short phrase,no style, no sketch/drawing . 
No explanation, no reasoning.


"""
    image_path = inputs["image_path"]
    image_b64 = encode_image(image_path)
    
    messages = [
    {"role": "user",
     "content": [
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": f"data:image/jpeg;base64,{image_b64}"}
     ]}
]



    result = sketch_descriptor_llm.invoke(messages)
    return result.content.strip()

In [23]:
test_output = get_sketch_desc_prompt.run({
    "inputs": {
        "image_path": f"{sketch_path}"
    }
})

In [24]:
test_output

'A small penguin with closed eyes and a smile.'